<a href="https://colab.research.google.com/github/JOk3r01001/Pipeline-approach-for-fracture-classification-using-YOLO-and-Keras/blob/main/YOLO_mask_export.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import json
from google.colab import drive

In [ ]:
import os
import json
from google.colab import drive

#Připojení Google Disku do prostředí Colab pro přístup k souborům
drive.mount('/content/drive')

#Definice cest
json_path = "/content/drive/MyDrive/BK2.0/FracAtlas2/Annotations/COCO JSON/COCO_fracture_masks.json"
output_dir = "/content/drive/MyDrive/BK2.0/FracAtlas2/Annotations/YOLO_Segmentation"

os.makedirs(output_dir, exist_ok=True)

with open(json_path, 'r') as f:
  coco_data = json.load(f)

#1 Vytvoření pomocného slovníku (ID obrázku -> jméno, šířka, výška)
img_map = {img['id']: (img['file_name'], img['width'], img['height']) for img in coco_data['images']}

print(f" conversion {len(coco_data['annotations'])} number of annotations...")

#2 Iterace přes všechny anotace v COCO souboru
for an in coco_data['annotations']:
    img_id = an['image_id']
    if img_id not in img_map: continue

    #Získání metadat snímků z předpřipravené mapy
    f_name, w, h = img_map[img_id]

#Zpracování segmentačních polygonů pro YOLO formát
    if 'segmentation' in an and an['segmentation']:
        txt_name = os.path.splitext(f_name)[0] + ".txt"
        save_path = os.path.join(output_dir, txt_name)

        with open(save_path, 'a') as f:
            for poly in an['segmentation']:
                #Normalizace dělení souřadnic x,y rozměrem obrázku
                normalized = []
                for i in range(0, len(poly), 2):
                    nx = poly[i] / w #x
                    ny = poly[i+1] / h #y
                    normalized.append(f"{nx:.6f} {ny:.6f}") #zaokrouhlení na 6des čísel

                #Zápis: 0 třída + souřadnice x1 y1 x2 y2 ...
                if normalized:
                    f.write(f"0 {' '.join(normalized)}\n")

print(f" FINISHED '{output_dir}' ")